# 86. SQL Query Writing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/11-domain-specific/86_sql_query_writing.ipynb)

**Category**: Domain-Specific Techniques  **Technique #86**

---

SQL Query Writing with LLMs involves generating database queries from natural language descriptions, optimizing existing queries, and explaining query logic.

## 📋 Description

SQL query generation helps:
- Convert business questions into database queries
- Write complex JOINs, subqueries, and CTEs
- Optimize slow-performing queries
- Generate DDL statements (CREATE, ALTER)
- Create views and stored procedures
- Explain query execution plans

**Best for**: Data analysts, backend developers, database administrators, business intelligence

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│  1. SCHEMA: Provide table structures and relationships      │
│     ↓                                                       │
│  2. CONTEXT: Explain business logic and data meaning        │
│     ↓                                                       │
│  3. REQUIREMENT: State what data you need                   │
│     ↓                                                       │
│  4. CONSTRAINTS: Performance, security, style preferences   │
│     ↓                                                       │
│  5. OUTPUT: Query with explanations and alternatives        │
└─────────────────────────────────────────────────────────────┘
```

**Schema Format:**
```sql
Table: users
- id (PK, INT)
- email (VARCHAR, UNIQUE)
- created_at (TIMESTAMP)

Table: orders
- id (PK, INT)
- user_id (FK, INT)
- total (DECIMAL)
- status (ENUM: pending, completed, cancelled)
```

## 🛠️ Setup

In [ ]:
# Install required packages
!pip install openai -q

import os
from getpass import getpass
from openai import OpenAI

# Get API key securely
api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

print("✓ Setup complete!")

## 💡 Basic Example

In [ ]:
def generate_sql(prompt, model="gpt-4"):
    """Generate SQL queries using OpenAI API."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are an expert SQL developer. Write optimized, secure queries with proper indexing hints and explain the query logic."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=1000
    )
    return response.choices[0].message.content

# Basic example: Simple SELECT with schema
basic_prompt = """
Given this schema:

Table: employees
- id (INT, PK)
- name (VARCHAR)
- department (VARCHAR)
- salary (DECIMAL)
- hire_date (DATE)

Write a query to find the top 5 highest-paid employees in the 'Engineering' department,
hired after 2020-01-01. Include their name, salary, and hire date.
"""

print("=== BASIC SQL GENERATION ===")
sql = generate_sql(basic_prompt)
print(sql)

## 🌍 Real-World Example

In [ ]:
# Real-world: Complex e-commerce analytics query
real_world_prompt = """
E-commerce Database Schema:

Table: customers
- customer_id (INT, PK)
- email (VARCHAR)
- registration_date (DATE)
- country (VARCHAR)

Table: orders
- order_id (INT, PK)
- customer_id (INT, FK)
- order_date (TIMESTAMP)
- status (ENUM: 'pending', 'shipped', 'delivered', 'cancelled')
- total_amount (DECIMAL)

Table: order_items
- item_id (INT, PK)
- order_id (INT, FK)
- product_id (INT, FK)
- quantity (INT)
- unit_price (DECIMAL)

Table: products
- product_id (INT, PK)
- product_name (VARCHAR)
- category (VARCHAR)
- cost_price (DECIMAL)

Task: Create a monthly sales report query that shows:
1. Month and year
2. Total revenue
3. Total orders
4. Average order value
5. Top selling category
6. Customer retention rate (customers who made 2+ orders)

Requirements:
- Use CTEs for readability
- Filter out cancelled orders
- Include only last 12 months
- Optimize for performance with proper indexing hints
- Handle NULL values appropriately
"""

print("\n=== REAL-WORLD SQL GENERATION ===")
complex_sql = generate_sql(real_world_prompt)
print(complex_sql)

## ⚠️ Failure Case

In [ ]:
# Poor prompt without schema context
poor_prompt = "Get all users who bought something last month."

print("=== POOR PROMPT (AMBIGUOUS) ===")
poor_sql = generate_sql(poor_prompt)
print(poor_sql)

print("\n" + "="*50)
print("PROBLEMS:")
print("="*50)
print("❌ No schema provided - table/column names guessed")
print("❌ 'Last month' is ambiguous - which timezone?")
print("❌ No definition of 'bought' - order status?")
print("❌ Output columns not specified")

print("\n=== IMPROVED PROMPT ===")
improved_prompt = """
Schema:
Table: users (user_id, email, created_at)
Table: purchases (purchase_id, user_id, purchase_date, amount, status)

Task: Find users who made at least one purchase with status='completed'
in the previous calendar month (e.g., if today is March 15, use February 1-28).

Return: user_id, email, total_spent, purchase_count
Order by: total_spent DESC
"""

better_sql = generate_sql(improved_prompt)
print(better_sql)

## 📊 Benchmark

| Model | SQL Accuracy | Complex JOINs | Optimization | Best For |
|-------|-------------|---------------|--------------|----------|
| GPT-4 | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Complex analytics |
| GPT-3.5 | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | Simple queries |
| Claude 3 | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Query explanation |
| CodeLlama | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | Open source |
| Text2SQL (BIRD) | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | Academic benchmark |

**Spider Benchmark (Text-to-SQL):**
- GPT-4: 85.3% execution accuracy
- Claude 3 Opus: 83.7% execution accuracy
- DIN-SQL + GPT-4: 91.2% execution accuracy

## 🎮 Interactive Playground

In [ ]:
# Interactive SQL playground
your_schema = """
[DESCRIBE YOUR TABLES HERE]

Example:
Table: posts (post_id, user_id, title, content, created_at, views)
Table: comments (comment_id, post_id, user_id, content, created_at)
Table: users (user_id, username, join_date)
"""

your_question = """
[YOUR NATURAL LANGUAGE QUESTION]

Example: Find the top 10 most engaged posts (by comment count) from users
who joined in the last 6 months, excluding posts with fewer than 5 comments.
"""

combined_prompt = f"""
Schema:
{your_schema}

Task: {your_question}
"""

# Uncomment to run:
# result = generate_sql(combined_prompt)
# print(result)

## 💡 Tips & Tricks

### SQL-Specific Prompting Tips

1. **Always provide schema** - Include table structures, data types, and relationships
2. **Use consistent naming** - Stick to either snake_case or camelCase
3. **Specify SQL dialect** - PostgreSQL, MySQL, SQL Server, SQLite have differences
4. **Include sample data** - 2-3 example rows help clarify data types
5. **Mention indexes** - Ask for index recommendations for large tables

### Security Best Practices

```sql
-- ❌ NEVER: Concatenate user input directly
WHERE id = ' + user_input + '

-- ✅ ALWAYS: Use parameterized queries
WHERE id = ?
```

### Performance Optimization

Ask the LLM to:
- Avoid SELECT *
- Use appropriate indexes
- Consider query execution order
- Use EXPLAIN ANALYZE for optimization hints

### Prompt Template

```
Database: [PostgreSQL/MySQL/etc]

Schema:
Table: [name]
- [column] ([type], [constraints])
- [FK relationships]

Indexes:
- [index_name] on [columns]

Task: [natural language description]

Requirements:
- [specific requirements]
- [performance constraints]
- [output format]
```

## 📚 References

1. [Spider: Yale Text-to-SQL Dataset](https://yale-lily.github.io/spider)
2. [BIRD Benchmark](https://bird-bench.github.io/)
3. [PostgreSQL Documentation](https://www.postgresql.org/docs/)
4. [SQL Style Guide by Simon Holywell](https://www.sqlstyle.guide/)
5. [Poosamani et al. (2023) - Text-to-SQL](https://arxiv.org/abs/2306.00739)